# 1 - Process trajectories (template)

Process dill track files into filtered parquet using shared helpers. Update config paths before running.

In [ ]:
from pathlib import Path
import pandas as pd
import glob

import bb_metrics
from bb_metrics.config import berlin2025 as cfg  # replace with your config
from bb_metrics import trajectories as traj

bb_metrics.set_config(cfg)


In [ ]:
# Load calibration/tag inputs
df_cornerpoints = pd.read_csv(cfg.saved_output_dir / 'df_cornerpoints.csv')
df_px_per_cm = pd.read_csv(cfg.saved_output_dir / 'df_px_per_cm.csv')
dftags_path = cfg.saved_output_dir / 'dftags.csv'
dftags = pd.read_csv(dftags_path) if dftags_path.exists() else None
dftags = dftags if dftags is not None and len(dftags) > 0 else None  # no filter if empty/absent

trackdir = cfg.trackdir
outdir = cfg.traj_outdir
dillfiles = sorted(Path(trackdir).glob('*.dill'))
len(dillfiles)


In [ ]:
# Process (parallel)
results = traj.process_files(
    dillfiles,
    df_cornerpoints,
    df_px_per_cm,
    dftags,
    update=True,
    confidence_threshold=0.8,
    min_num_obs_in_1hrs_tracked=15,
    max_speed=5.0,
    outdir=outdir,
    cam_hive_map=cfg.cam_hive_map,
    max_workers=4,
)
pd.Series([r[1] for r in results]).value_counts()
